In [0]:
from pyspark.sql.functions import *
data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),  # updated record
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),  # invalid amount
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)   # duplicate
]
columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]
df=spark.createDataFrame(data,columns)
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df=df.dropDuplicates()
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,null,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,null,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,-45000,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
df=df.withColumn("amount",
        when(col("amount")<0,0).otherwise(col("amount")))
df=df.withColumn("city",
        when(col("city").isNull(),"Unknown").otherwise(col("city")))
df=df.fillna({
    "amount":0
})
df=df.withColumn("date",to_date("date","yyyy-mm-dd"))
df.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-03,20000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,0,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


In [0]:
dbutils.fs.rm("/Volumes/workspace/default/practice/medallion_assignment", True)


True

In [0]:
df.write.format("delta").mode("overwrite").save("/Volumes/workspace/default/practice/medallion_assignment")

##Sales analysis

In [0]:
order_product=df.groupBy("product").sum("amount").orderBy(sum("amount"))


In [0]:
order_category=df.groupBy("category").sum("amount").orderBy(sum("amount"))


In [0]:
order_city=df.groupBy("city").sum("amount").orderBy(sum("amount"))


In [0]:
order_cust=df.groupBy("customer_id").agg(count("order_id"))
order_cust.display()

customer_id,count(order_id)
C001,3
C002,2
C003,1
C004,1
C005,1
C006,1
C007,1


In [0]:
top_cust=df.groupBy("customer_id").agg(sum("amount")).orderBy(sum("amount").desc())
top_cust.display(

)

customer_id,sum(amount)
C001,92000
C003,55000
C002,18000
C007,15000
C004,8000
C005,3000
C006,0


In [0]:
from pyspark.sql.window import Window
win=Window.partitionBy("order_id").orderBy(col("date").desc())
latest_date=df.withColumn("rank",dense_rank().over(win))
latest_date=latest_date.filter(col("rank")==1).drop("rank")
latest_date.display()

order_id,customer_id,product,category,city,date,amount,quantity
101,C001,Laptop,Electronics,Hyderabad,2024-01-01,50000,1
102,C002,Mobile,Electronics,Chennai,2024-01-02,0,2
103,C001,Tablet,Electronics,Hyderabad,2024-01-07,22000,1
104,C003,Laptop,Electronics,Delhi,2024-01-04,55000,1
105,C002,Mobile,Electronics,Chennai,2024-01-05,18000,1
106,C004,Watch,Accessories,Mumbai,2024-01-06,8000,1
107,C005,Headphones,Accessories,Unknown,2024-01-08,3000,1
108,C006,Laptop,Electronics,Bangalore,2024-01-09,0,1
109,C007,Mobile,Electronics,Chennai,2024-01-10,15000,2


##Tasks

In [0]:
total_orders=df.groupBy("customer_id").agg(count("order_id")).orderBy(count("order_id").desc())
total_orders.display()
total_spending=df.groupBy("customer_id").agg(sum("amount")).orderBy(sum("amount").desc())

customer_id,count(order_id)
C001,3
C002,2
C003,1
C004,1
C005,1
C006,1
C007,1


In [0]:
top_sell=df.groupBy("product").agg(sum("amount")).orderBy(sum("amount").desc()).limit(1)
top_sell.display()

product,sum(amount)
Laptop,105000


In [0]:
top_cust=df.groupBy("customer_id").agg(sum("amount")).orderBy(sum("amount").desc()).limit(1)
top_cust.display()

customer_id,sum(amount)
C001,92000
